# PREFFECT: a worked example

The same steps as the documentation's [worked example](../docs_src/vignette.md), as a notebook: write a small simulated cohort, check it, fit a model, apply it to data the model has never seen, and score what it filled in. About three minutes end to end.

Each cell calls the command line from Python, so what runs here is exactly what `preffect check`, `preffect train` and `preffect apply` run. Install PREFFECT first (see the README), then run the cells in order. Everything is written into the folder the notebook runs in.

In [ ]:
from preffect.cli import main

# A small simulated cohort: three splits with no sample in common, for each of the three models.
assert main(["example-data", "--out", "vignette_data"]) == 0

## 1. Check the data first

`check` reads the counts without fitting anything. The last line it prints says whether any training sample reappears in the validation set, which is how the problem with version 0.1's vignettes would have been caught.

In [ ]:
assert main(["check", "--train", "vignette_data/simple/train",
             "--validation", "vignette_data/simple/validation"]) == 0

## 2. Fit

`--patience 5` stops once the validation loss has not fallen for five epochs, and the model written is the best epoch's. `--seed` makes the run repeatable; `--threads 4` suits a small example.

In [ ]:
assert main(["train", "--train", "vignette_data/simple/train",
             "--validation", "vignette_data/simple/validation",
             "--out", "model.pt",
             "--epochs", "80", "--batch-size", "40", "--seed", "0", "--threads", "4",
             "--patience", "5"]) == 0

## 3. Apply it to data it has never seen

The test split was never seen during fitting and never used to decide when to stop, so what comes out is a genuine held-out result.

In [ ]:
assert main(["apply", "--model", "model.pt",
             "--data", "vignette_data/simple/test",
             "--out", "cleaned.h5ad", "--threads", "4"]) == 0

### What came out

`cleaned.h5ad` carries the adjusted counts in `X`, the predicted proportions in `layers['composition']`, and the latent summary of each sample in `obsm['latent']`.

In [ ]:
import anndata as ad

cleaned = ad.read_h5ad("cleaned.h5ad")
print(cleaned)
print("latent summary:", cleaned.obsm["latent"].shape)

## 4. Score what it filled in

Hide a fifth of the training entries, fit again on what is left, and score the hidden entries. The error the run prints is the distance from the count that was observed, and most of it is noise rather than error: a count is one draw from a negative binomial, and nothing can predict the draw. This simulated cohort also carries the mean each count was drawn from, in `layers['true_expected_counts']`, so the two can be told apart. No real cohort has that layer, and PREFFECT will not invent one.

In [ ]:
assert main(["train", "--train", "vignette_data/simple/train",
             "--validation", "vignette_data/simple/validation",
             "--out", "imputing.pt", "--hide", "0.2",
             "--epochs", "80", "--batch-size", "40", "--seed", "0", "--threads", "4",
             "--patience", "5"]) == 0

In [ ]:
from pathlib import Path

import numpy as np
import torch

from preffect.checkpoint import load_checkpoint
from preffect.inference import imputation_error, infer
from preffect.workflow import build_model, hide_entries, read_split, to_model_data

checkpoint = load_checkpoint("imputing.pt")
model = build_model(
    checkpoint.model_config.kind, checkpoint.n_genes, checkpoint.model_config, 1
)
model.load_state_dict(checkpoint.model_state)
model.eval()

# The same entries the run hid: the fraction and the seed are what choose them.
tissues = read_split(Path("vignette_data/simple/train"), 1)
data = hide_entries(to_model_data(tissues, checkpoint.model_config, False), 0.2, 0)
truth = torch.from_numpy(
    np.asarray(tissues[0].layers["true_expected_counts"], dtype=np.float32)
)

scores = imputation_error(
    infer(model, data).expected_counts, truth,
    hidden=torch.from_numpy(data.mask.hidden), kind="mean_relative",
)
print("against the count each entry was drawn from:", round(scores["hidden"], 4))

## Next

- The [graph model](../docs_src/vignette.md#the-graph-model) on `vignette_data/single/`, which carries a sample-sample network with real edges.
- [Using it](../docs_src/guide.md) for imputation, clustering and figures from Python.
- [The command line](../docs_src/command-line.md) for every option.